# AVG Master Data - Restructured

Run all cells in order. Uses local paths by default.

## Google Colab

1. Run **Install dependencies** (next cell).
2. Run **Colab paths + Drive** (cell after that). Set `MOUNT_GOOGLE_DRIVE = False` if your data is only under `/content` (then edit `COLAB_DATA_DIR` or upload files there).
3. Put the same Excel/CSV layout as on your Mac in **`MyDrive/Project 005 Data/`** (or change `COLAB_DATA_DIR` in that cell).

On Drive, `read_excel` can hit FUSE errors; this notebook defines **`read_excel_colab_safe`** and uses it for workbook reads where patched.

**If you see `ImportError: cannot import name '_center' from 'numpy._core.umath'`**, Colab's pre-installed numpy was broken by an earlier `pip install -U`. Recover with:

```
!pip install --force-reinstall -q "numpy==2.0.2" "pandas==2.2.2"
```

then **Runtime → Restart session** and run this notebook from the top. The install cell below no longer upgrades numpy/pandas/torch and will stop the notebook if the runtime needs a restart.


In [ ]:
# Install dependencies.
# Colab (esp. GPU runtime w/ RAPIDS) pins numpy<2.1 and pandas<2.4. Any `pip install -U`
# can desync wheels and break `numpy._core.umath._center` (imported by numpy.strings,
# xgboost, sklearn, etc.). We therefore:
#   1) Detect a broken/split numpy,
#   2) Delete shadow dist dirs pip left behind (`~numpy*`, `~orch*`, half-removed numpy),
#   3) Surgically reinstall numpy==2.0.2 and pandas==2.2.2 (compatible with RAPIDS),
#   4) Halt the notebook with a clear "Restart session" message.
# We never use `-U` on Colab and never touch torch here; the GPU runtime ships its own.
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

_IN_COLAB = "google.colab" in sys.modules
_USE_TORCH = False
_USE_STATS = True


def _pip(*args):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *args],
        check=True,
    )


def _has(module: str) -> bool:
    try:
        importlib.import_module(module)
        return True
    except Exception:
        return False


def _numpy_healthy() -> bool:
    # If numpy._core.umath._center is missing, the numpy install is split across
    # incompatible wheels and Python cannot recover in the current session.
    try:
        from numpy._core import umath as _um
        import numpy.strings  # noqa: F401  same path xgboost/sklearn trigger
        return hasattr(_um, "_center")
    except Exception:
        return False


def _colab_cleanup_and_repair_numpy() -> None:
    # Wipe pip's abandoned shadow installs (names start with '~') and any half-removed
    # numpy dir/dist-info — these are what make `pip install --force-reinstall numpy`
    # silently fail to fully overwrite the old files.
    import glob
    site_roots = [Path(p) for p in sys.path if p.endswith("dist-packages") or p.endswith("site-packages")]
    site_roots = [p for p in site_roots if p.is_dir()]
    for root in site_roots:
        for pat in ("~*", "numpy", "numpy-*.dist-info", "numpy.libs"):
            for hit in glob.glob(str(root / pat)):
                try:
                    if Path(hit).is_dir():
                        shutil.rmtree(hit, ignore_errors=True)
                    else:
                        Path(hit).unlink(missing_ok=True)
                except Exception:
                    pass
    # Reinstall a numpy/pandas pair that's compatible with Colab GPU (RAPIDS: numpy<2.1).
    _pip("--no-deps", "--force-reinstall", "numpy==2.0.2")
    _pip("--force-reinstall", "pandas==2.2.2")


_COLAB_RESTART_MSG = (
    "Colab runtime was repaired (numpy/pandas reinstalled to versions compatible with "
    "RAPIDS / GPU runtime). Please **Runtime -> Restart session** and then run this "
    "notebook from the top."
)

if _IN_COLAB:
    if not _numpy_healthy():
        _colab_cleanup_and_repair_numpy()
        raise RuntimeError(_COLAB_RESTART_MSG)
    _needed = []
    for _mod, _pkg in [
        ("xgboost", "xgboost"),
        ("openpyxl", "openpyxl"),
        ("tqdm", "tqdm"),
        ("sklearn", "scikit-learn"),
        ("matplotlib", "matplotlib"),
    ]:
        if not _has(_mod):
            _needed.append(_pkg)
    if _USE_TORCH and not _has("torch"):
        _needed.append("torch")
    if _USE_STATS and not _has("statsmodels"):
        _needed.append("statsmodels")
    if _needed:
        _pip(*_needed)  # intentionally NO -U
        raise RuntimeError(
            "Installed missing packages on Colab: " + ", ".join(_needed) + ". "
            "Please **Runtime -> Restart session**, then run the notebook from the top."
        )
    print("Colab env OK - no pip actions needed.")
else:
    _PKGS = ["pandas", "numpy", "matplotlib", "openpyxl", "scikit-learn", "tqdm", "xgboost"]
    if _USE_TORCH:
        _PKGS = ["torch", *_PKGS]
    if _USE_STATS:
        _PKGS.append("statsmodels")
    _pip("-U", *_PKGS)
    print("pip OK | Colab: False")


In [ ]:
# --- Colab: paths, optional Drive mount, safe Excel reads ---
import os
import shutil
import sys
from pathlib import Path

import pandas as pd

IN_COLAB = "google.colab" in sys.modules
MOUNT_GOOGLE_DRIVE = True  # False if you only use /content uploads
COLAB_DATA_DIR = Path("/content/drive/MyDrive/Project 005 Data")


def _marker_exists(p: Path, name: str = "Master_Data_AVG_version_update.xlsx") -> bool:
    try:
        return (p / name).is_file()
    except OSError:
        return False


def resolve_project005_base_dir() -> Path:
    if IN_COLAB:
        if COLAB_DATA_DIR.is_dir() and _marker_exists(COLAB_DATA_DIR):
            return COLAB_DATA_DIR.resolve()
        alt = Path("/content")
        if _marker_exists(alt):
            return alt.resolve()
        return COLAB_DATA_DIR.resolve()
    here = Path.cwd().resolve()
    if _marker_exists(here):
        return here
    fb = (Path.home() / "Downloads" / "Project 005 Data").resolve()
    if fb.is_dir() and _marker_exists(fb):
        return fb
    return here


if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive mount skipped or failed:", exc)

BASE_DIR = resolve_project005_base_dir()
if IN_COLAB and BASE_DIR.is_dir():
    try:
        os.chdir(BASE_DIR)
    except OSError:
        pass


def read_excel_colab_safe(path, **kwargs):
    """Read Excel; on Colab copy from Drive to /tmp first (reduces FUSE transport errors)."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    resolved = path.expanduser().resolve()
    sp = os.fspath(resolved)
    if IN_COLAB and sp.startswith("/content/drive/"):
        local = Path("/tmp") / f"_colab_safe_{resolved.name}"
        shutil.copyfile(resolved, local)
        try:
            return pd.read_excel(local, **kwargs)
        finally:
            try:
                local.unlink(missing_ok=True)
            except OSError:
                pass
    return pd.read_excel(resolved, **kwargs)


print("IN_COLAB:", IN_COLAB, "| BASE_DIR:", BASE_DIR)


In [ ]:
# --- Colab NumPy guard (before heavy imports): numpy._core.umath must expose _center. ---
import sys as _sys
if "google.colab" in _sys.modules:
    try:
        from numpy._core import umath as _np_um
        import numpy.strings  # noqa: F401  same path xgboost/sklearn trigger
        _np_ok = hasattr(_np_um, "_center")
    except Exception:
        _np_ok = False
    if not _np_ok:
        import glob as _glob, shutil as _shutil, subprocess as _sp
        from pathlib import Path as _Path
        _roots = [_Path(p) for p in _sys.path if p.endswith("dist-packages") or p.endswith("site-packages")]
        for _root in [_r for _r in _roots if _r.is_dir()]:
            for _pat in ("~*", "numpy", "numpy-*.dist-info", "numpy.libs"):
                for _hit in _glob.glob(str(_root / _pat)):
                    try:
                        if _Path(_hit).is_dir():
                            _shutil.rmtree(_hit, ignore_errors=True)
                        else:
                            _Path(_hit).unlink(missing_ok=True)
                    except Exception:
                        pass
        _sp.run([_sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", "numpy==2.0.2"], check=True)
        _sp.run([_sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "pandas==2.2.2"], check=True)
        raise RuntimeError(
            "Colab numpy was broken and has been reinstalled (2.0.2). "
            "Please **Runtime -> Restart session**, then run the notebook from the top."
        )



In [1]:
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", None)



INPUT_FILE = os.path.join(BASE_DIR, "Master_Data_Base_version_updated.xlsx")
OUTPUT_FILE = os.path.join(BASE_DIR, "Master_Data_AVG_version_update.xlsx")

# If file not in cwd, try notebook directory (e.g. when opened from Downloads/Project 005 Data)
if not os.path.exists(INPUT_FILE):
    for d in [os.path.expanduser("~/Downloads/Project 005 Data"), os.path.dirname(os.path.abspath("."))]:
        p = os.path.join(d, "Master_Data_Base_version_updated.xlsx")
        if os.path.exists(p):
            BASE_DIR = d
            INPUT_FILE = p
            OUTPUT_FILE = os.path.join(d, "Master_Data_AVG_version_update.xlsx")
            break
print(f"Using BASE_DIR: {BASE_DIR}")


Using BASE_DIR: /Users/andyyang/Downloads/Project 005 Data


In [2]:
def get_column_intervals(df):
    """For each numeric column, return column name and max_gap_days."""
    idx = (
        df.index
        if isinstance(df.index, pd.DatetimeIndex)
        else pd.to_datetime(df["Date"])
        if "Date" in df.columns
        else pd.RangeIndex(len(df))
    )
    result = []
    for col in df.select_dtypes(include="number").columns:
        valid = df[col].notna()
        if valid.any():
            valid_dates = idx[valid].sort_values()
            if isinstance(valid_dates, pd.DatetimeIndex):
                diffs = valid_dates.to_series().diff().dropna()
                gaps = (diffs / pd.Timedelta(days=1)).astype(int)
                max_gap = int(gaps.max()) if len(gaps) > 0 else None
            else:
                max_gap = None
            result.append({"column": col, "max_gap_days": max_gap})
        else:
            result.append({"column": col, "max_gap_days": None})
    return pd.DataFrame(result)


In [3]:
# Load data and set datetime index
master_df_update = read_excel_colab_safe(INPUT_FILE)
master_df_update.reset_index(inplace=True, drop=True)
if "Date" in master_df_update.columns:
    master_df_update.index = pd.to_datetime(master_df_update["Date"])
    master_df_update.index.name = "Date"
master_df_update.head()


,Date,C5,China Manufacturing Purchasing Managers Index (Official)_PMI,Industrial Production China_% Yr/Yr,Industrial Production Japan_% Yr/Yr,Industrial Production S Korea_% Yr/Yr,"China Steel Production_,000 tonnes","Japan Steel Production_,000 tonnes","South Korea Steel Production_,000 tonnes","P.R. China BFI Production_,000 tonnes",...,steel margin cash profit with cost delayed 1 month for marginal producer of Rebar,steel margin cash profit with cost delayed 1 week for marginal producer of Rebar,steel margin cash profit with cost delayed 1 month for marginal producer of HRC,steel margin cash profit with cost delayed 1 week for marginal producer of HRC,steel margin cash profit with cost delayed 1 month for marginal producer of CRC,steel margin cash profit with cost delayed 1 week for marginal producer of CRC,Guinea Total Bauxite Exports,Guinea Bauxite Exports in Capes,CHN Imports from Guinea,CHN Imports from Guinea inCapes
Date,,,,,,,,,,,,,,,,,,,,,
2005-12-09,2005-12-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-12,2005-12-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-13,2005-12-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-14,2005-12-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-15,2005-12-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Optional: drop columns if they exist
column_drop = [
    "Crude Steel Production, Current Period (10,000 Tons)",
    "Brent Crude Oil Price, 1 Month Future_$/bbl",
    "Australia Seaborne Iron Ore Exports_Million Tonnes",
    "China Seaborne Iron Ore Imports_Million Tonnes",
]
existing_drop = [c for c in column_drop if c in master_df_update.columns]
if existing_drop:
    master_df_update.drop(columns=existing_drop, inplace=True)
print(f"Columns: {len(master_df_update.columns)}")


Columns: 180


In [5]:
# Columns to apply business day averages (by name)
col_avg_names = [
    "China Steel Production_,000 tonnes",
    "Japan Steel Production_,000 tonnes",
    "South Korea Steel Production_,000 tonnes",
    "P.R. China BFI Production_,000 tonnes",
    "Japan BFI Production_,000 tonnes",
    "R.O.Korea BFI Production_,000 tonnes",
    "China Seaborne Coal Imports_Million Tonnes",
    "China Seaborne Steel Products Exports_Million Tonnes",
    "Iron Ore Production (10,000 tonnes)",
    "Steel Products Production, Current Period (10,000 Tons)",
    "Rebar Production, Current Period (10,000 Tons)",
    "Pig Iron Production, Current Period (10,000 Tons)",
    "Australia Seaborne Iron Ore Exports_Million Tonnes.1",
    "Brazil Seaborne Iron Ore Exports_Million Tonnes",
    "China Seaborne Iron Ore Imports_Million Tonnes.1",
    "South Korea Seaborne Iron Ore Imports_Million Tonnes",
    "Japan Seaborne Iron Ore Imports_Million Tonnes",
    "Taiwan Seaborne Iron Ore Imports_Million Tonnes",
    "Guinea Total Bauxite Exports",
    "Guinea Bauxite Exports in Capes",
    "CHN Imports from Guinea",
    "CHN Imports from Guinea inCapes",
]
col_avg = [c for c in col_avg_names if c in master_df_update.columns]
col_not_month_idx = [0, 22, 55, 58] + list(range(43, 46)) + list(range(91, 94))
col_not_month = col_not_month_idx + [i for i, c in enumerate(master_df_update.columns) if c in col_avg]
col_not_month = [i for i in col_not_month if i < len(master_df_update.columns)]
print(f"col_avg: {len(col_avg)} cols, col_not_month: {len(col_not_month)} cols")


col_avg: 22 cols, col_not_month: 32 cols


In [6]:
# 1. Business day averages for col_avg
def bdays_in_month(dt):
    return len(pd.bdate_range(dt.replace(day=1), dt + pd.offsets.MonthEnd(0)))

for col_name in col_avg:
    monthly_mean = master_df_update.groupby(master_df_update.index.to_period("M"))[col_name].transform("mean")
    bdays = master_df_update.index.to_series().apply(bdays_in_month)
    master_df_update[col_name] = monthly_mean / bdays
print(f"Business day averages applied to {len(col_avg)} columns.")


Business day averages applied to 22 columns.


In [7]:
# 2. Target/source logic: cols 24,32,36,40 from 25,33,37,41 (Feb 1st anchor)
target_cols = master_df_update.columns[[24, 32, 36, 40]] if all(i < len(master_df_update.columns) for i in [24, 32, 36, 40]) else []
source_cols = master_df_update.columns[[25, 33, 37, 41]] if all(i < len(master_df_update.columns) for i in [25, 33, 37, 41]) else []

if len(target_cols) == 4 and len(source_cols) == 4:
    pairs = list(zip(target_cols, source_cols))
    feb_1st_mask = (master_df_update.index.month == 2) & (master_df_update.index.day == 1)
    feb_1st_data = master_df_update.loc[feb_1st_mask]
    anchor_map = {}
    for year in feb_1st_data.index.year.unique():
        subset = feb_1st_data[feb_1st_data.index.year == year]
        if len(subset) > 0:
            anchor_map[year] = subset[source_cols].iloc[0].to_dict()

    for t_col, s_col in pairs:
        for year in anchor_map:
            jan_mask = (master_df_update.index.year == year) & (master_df_update.index.month == 1)
            feb_mask = (master_df_update.index.year == year) & (master_df_update.index.month == 2)
            b_days_jan = np.busday_count(f"{year}-01-01", f"{year}-02-01")
            b_days_feb = np.busday_count(f"{year}-02-01", f"{year}-03-01")
            total_b_days = b_days_jan + b_days_feb
            source_val = anchor_map[year].get(s_col, 0)
            source_val = source_val if pd.notna(source_val) else 0
            feb_sum = master_df_update.loc[feb_mask, t_col].sum()
            feb_sum = feb_sum if pd.notna(feb_sum) else 0
            if feb_sum > 0:
                jan_fill = (source_val - feb_sum) / b_days_jan
                master_df_update.loc[jan_mask & (master_df_update[t_col].isna() | (master_df_update[t_col] == 0)), t_col] = jan_fill
            else:
                combined_fill = source_val / total_b_days
                master_df_update.loc[(jan_mask | feb_mask) & (master_df_update[t_col].isna() | (master_df_update[t_col] == 0)), t_col] = combined_fill

    master_df_update = master_df_update[master_df_update.index.weekday < 5]
    print("Target/source logic applied (weekdays only).")
else:
    print("Skipped target/source: column indices out of range.")


Target/source logic applied (weekdays only).


In [8]:
# 3. Forward fill remaining columns with max_gap_days (cap at 31 if > 100)
_ci = get_column_intervals(master_df_update)
max_gap_map = dict(zip(_ci["column"], _ci["max_gap_days"]))

remaining_idx = [i for i in range(len(master_df_update.columns)) if master_df_update.columns[i] not in col_avg]
for i in remaining_idx:
    name = master_df_update.columns[i]
    max_gap = max_gap_map.get(name)
    limit = 31 if (max_gap is None or pd.isna(max_gap) or max_gap > 100) else int(max_gap)
    master_df_update[name] = master_df_update[name].ffill(limit=limit)
print(f"Forward fill applied to {len(remaining_idx)} remaining columns.")


Forward fill applied to 158 remaining columns.


In [9]:
# 4. Per-column fill logic for col_not_month
for i in range(len(master_df_update.columns)):
    name = master_df_update.columns[i]

    if i not in col_not_month:
        master_df_update[name] = master_df_update.groupby(master_df_update.index.to_period("M"))[name].transform(lambda x: x.ffill().bfill())
    elif i in range(43, 46):
        master_df_update[name] = master_df_update.groupby(master_df_update.index.to_period("M"))[name].transform("bfill")
    elif i != 0:
        last_val_idx = master_df_update[name].last_valid_index()
        if last_val_idx is not None:
            master_df_update.loc[:last_val_idx, name] = master_df_update.loc[:last_val_idx, name].ffill()
            master_df_update.loc[last_val_idx:, name] = master_df_update.loc[last_val_idx:, name].ffill(limit=7)
print("Per-column fill logic applied.")


Per-column fill logic applied.


In [10]:
master_df_update.head(20)


,Date,C5,China Manufacturing Purchasing Managers Index (Official)_PMI,Industrial Production China_% Yr/Yr,Industrial Production Japan_% Yr/Yr,Industrial Production S Korea_% Yr/Yr,"China Steel Production_,000 tonnes","Japan Steel Production_,000 tonnes","South Korea Steel Production_,000 tonnes","P.R. China BFI Production_,000 tonnes",...,steel margin cash profit with cost delayed 1 month for marginal producer of Rebar,steel margin cash profit with cost delayed 1 week for marginal producer of Rebar,steel margin cash profit with cost delayed 1 month for marginal producer of HRC,steel margin cash profit with cost delayed 1 week for marginal producer of HRC,steel margin cash profit with cost delayed 1 month for marginal producer of CRC,steel margin cash profit with cost delayed 1 week for marginal producer of CRC,Guinea Total Bauxite Exports,Guinea Bauxite Exports in Capes,CHN Imports from Guinea,CHN Imports from Guinea inCapes
Date,,,,,,,,,,,,,,,,,,,,,
2005-12-09,2005-12-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-12,2005-12-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-13,2005-12-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-14,2005-12-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-15,2005-12-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-16,2005-12-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-19,2005-12-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-20,2005-12-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-21,2005-12-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Export
master_df_update.to_excel(OUTPUT_FILE, index=True)
print(f"Saved to {OUTPUT_FILE}")


Saved to /Users/andyyang/Downloads/Project 005 Data/Master_Data_AVG_version_update.xlsx


In [12]:
# Column intervals (optional)
column_intervals = get_column_intervals(master_df_update)
column_intervals


,column,max_gap_days
0,C5,3
1,China Manufacturing Purchasing Managers Index ...,3
2,Industrial Production China_% Yr/Yr,3
3,Industrial Production Japan_% Yr/Yr,3
4,Industrial Production S Korea_% Yr/Yr,3
5,"China Steel Production_,000 tonnes",3
6,"Japan Steel Production_,000 tonnes",3
7,"South Korea Steel Production_,000 tonnes",3
8,"P.R. China BFI Production_,000 tonnes",3
9,"Japan BFI Production_,000 tonnes",3
